# 🚀 SPV TPU: Google Colab TPU 最適化 3D FDTD + 本家 Harminv CLI 共振モード解析

本ノートブックは `spv_gpu` を **Google Colab TPU（v5e-1 / v6e-1 / v2-8 等）** 向けに最適化した `spv_tpu` パッケージのランチャーです。
コードは GitHub リポジトリ [`kyouoku/colab_GPU`](https://github.com/kyouoku/colab_GPU.git) から自動取得します。

### TPU 最適化の主なポイント（対 `spv_gpu`）
1. **TPU VPU `(8, 128)` タイル整合 `(Nz, Ny, Nx)` レイアウト**: `Nx` を 128 の倍数、`Ny` を 8（または `8 × シャード数`）の倍数へ自動アラインし、XLA の暗黙パディングによる HBM・帯域ロスを排除
2. **Zero-`jnp.roll` スライス差分 ＋ Yee 厳密鏡像境界**: `jnp.roll` による HBM コピー/転置を全廃し、`pz=+1` (PMC) の半格子結合を厳密化
3. **境界スラブ CFS-CPML（既定）**: 3D 全体への18配列確保を廃止し、境界スラブのみに補助場を配置して HBM 消費を抑えつつ高 Q 共振器の広帯域吸収を実現
4. **マルチチップ TPU `y` 軸空間分割 (`shard_map` + `lax.ppermute`)**: `v2-8` / `v5e-8` / `v6e-8` 等では自動的に領域分割を実行
5. **XLA 永続コンパイルキャッシュ ＋ ホスト多コア Harminv 並列化**: STEP 2.5 → STEP 3 の再コンパイルを回避し、TPU VM の全 vCPU で `harminv` を並列実行

> **💡 ヒント**: Google Colab のメニューから **「ランタイム > ランタイムのタイプを変更 > ハードウェアアクセラレータ: TPU (v5e-1 / v6e-1 等)」** を選択してから上から順に実行してください。

In [ ]:
# ==============================================================================
# 1. TPU 安全な環境セットアップ & 本家 Harminv CLI のインストール
#    ※ Colab プリインストールの jax / libtpu を壊さないよう上書きインストールしません
# ==============================================================================
import importlib.util
import os
import shutil
import subprocess
import sys

# 欠けている軽量依存パッケージのみ補完 (jax / fdtdx は上書きしない)
missing_pkgs = [p for p in ("pandas", "matplotlib", "numpy") if importlib.util.find_spec(p) is None]
if missing_pkgs:
    print(f"📦 Installing missing Python packages: {missing_pkgs} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs], check=False)

# JAX TPU 確認 (未導入時のみ jax[tpu] を案内・導入)
if importlib.util.find_spec("jax") is None:
    print("📦 JAX not found; installing jax[tpu] ...")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "jax[tpu]",
            "-f", "https://storage.googleapis.com/jax-releases/libtpu_releases.html",
        ],
        check=False,
    )

# XLA 永続コンパイルキャッシュ設定
XLA_CACHE_DIR = "/tmp/spv_tpu_xla_cache"
os.makedirs(XLA_CACHE_DIR, exist_ok=True)
os.environ["JAX_COMPILATION_CACHE_DIR"] = XLA_CACHE_DIR

# 本家 Harminv CLI バイナリのインストール (apt -> conda-forge フォールバック)
if shutil.which("harminv") is None:
    print("🔧 Installing upstream harminv CLI via apt-get...")
    subprocess.run(
        "apt-get update -qq && apt-get install -y -qq harminv",
        shell=True,
        capture_output=True,
        text=True,
    )
    if shutil.which("harminv") is None:
        conda_bin = shutil.which("mamba") or shutil.which("conda")
        if conda_bin:
            print(f"⚠️ apt install unavailable; trying {conda_bin} (conda-forge) fallback...")
            subprocess.run([conda_bin, "install", "-y", "-c", "conda-forge", "harminv"], check=False)

harminv_path = shutil.which("harminv")
if harminv_path:
    ver_out = "harminv CLI installed"
    for flag in ("-V", "--version", "-h"):
        res = subprocess.run([harminv_path, flag], capture_output=True, text=True, check=False)
        combined = ((res.stdout or "") + "\n" + (res.stderr or "")).strip()
        if combined:
            first_line = combined.splitlines()[0].strip()
            if "invalid option" not in first_line.lower() and "unrecognized option" not in first_line.lower():
                ver_out = first_line
                break
    print(f"✅ Harminv CLI ready: {ver_out} ({harminv_path})")
else:
    print("⚠️ Harminv CLI binary not found in PATH (Matrix-Pencil SVD fallback will be used).")

import jax
jax.config.update("jax_compilation_cache_dir", XLA_CACHE_DIR)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)

devs = jax.devices()
print(f"✅ JAX version: {jax.__version__} | Backend: {jax.default_backend()} | Devices: {len(devs)}x {getattr(devs[0], 'device_kind', devs[0])}")
print(f"🖥️ Host vCPUs: {os.cpu_count()} | XLA cache: {XLA_CACHE_DIR}")
for idx, d in enumerate(devs[:2]):
    try:
        ms = d.memory_stats()
        if ms and "bytes_limit" in ms:
            print(f"   Chip {idx} HBM limit: {ms['bytes_limit'] / (1024**3):.2f} GiB")
    except Exception:
        pass
if jax.default_backend() != "tpu":
    print("⚠️ 警告: TPU が検出されていません。Colab の「ランタイム > ランタイムのタイプを変更」から TPU を選択してください。")

## 2. Google Drive のマウントと GitHub リポジトリ (`https://github.com/kyouoku/colab_GPU.git`) の取得
GitHub リポジトリから最新の `spv_tpu` パッケージを取得し、Google Drive 上の出力先ディレクトリと XLA 永続キャッシュを設定します。

In [ ]:
# ==============================================================================
# 2. Google Drive マウント & GitHub リポジトリ (kyouoku/colab_GPU) の取得
# ==============================================================================
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/kyouoku/colab_GPU.git"
MOUNT_GOOGLE_DRIVE = True      # False にするとローカルの ./spv_tpu_results に保存
USE_DRIVE_XLA_CACHE = False    # True にするとコンパイルキャッシュも Drive に永続化

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DEFAULT_OUT_DIR = "/content/drive/MyDrive/spv_tpu_results"
        print(f"✅ Google Drive mounted. Default output_dir: {DEFAULT_OUT_DIR}")
    except Exception as e:
        DEFAULT_OUT_DIR = os.path.abspath("./spv_tpu_results")
        print(f"ℹ️ Running outside Colab or Drive mount skipped ({e}). Using {DEFAULT_OUT_DIR}")
else:
    DEFAULT_OUT_DIR = os.path.abspath("./spv_tpu_results")

if USE_DRIVE_XLA_CACHE and DEFAULT_OUT_DIR.startswith("/content/drive"):
    XLA_CACHE_DIR = "/content/drive/MyDrive/spv_tpu_xla_cache"
    os.makedirs(XLA_CACHE_DIR, exist_ok=True)
    import jax
    jax.config.update("jax_compilation_cache_dir", XLA_CACHE_DIR)
    print(f"💾 Persistent XLA cache enabled on Google Drive: {XLA_CACHE_DIR}")

if os.path.isdir("spv_tpu") and os.path.isfile("spv_tpu/config.py"):
    REPO_DIR = os.path.abspath(".")
    print(f"📂 Using existing local repository at: {REPO_DIR}")
elif os.path.isdir("../spv_tpu") and os.path.isfile("../spv_tpu/config.py"):
    REPO_DIR = os.path.abspath("..")
    print(f"📂 Using parent repository at: {REPO_DIR}")
else:
    REPO_DIR = "/content/colab_GPU" if os.path.isdir("/content") else os.path.abspath("./colab_GPU")
    if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
        print(f"📥 Cloning {REPO_URL} into {REPO_DIR} ...")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        print(f"🔄 Pulling latest changes in {REPO_DIR} ...")
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import spv_tpu
import spv_tpu.config
import spv_tpu.geometry.shapes
import spv_tpu.geometry.smoothing
import spv_tpu.geometry.structures
import spv_tpu.solver.symmetry
import spv_tpu.solver.tpu_runner
import spv_tpu.analysis.harminv_cli
import spv_tpu.analysis.select_modes
import spv_tpu.pipeline
import spv_tpu.validate_vs_meep
for mod in (
    spv_tpu.config, spv_tpu.geometry.shapes, spv_tpu.geometry.smoothing,
    spv_tpu.geometry.structures, spv_tpu.solver.symmetry, spv_tpu.solver.tpu_runner,
    spv_tpu.analysis.harminv_cli, spv_tpu.analysis.select_modes,
    spv_tpu.pipeline, spv_tpu.validate_vs_meep, spv_tpu,
):
    importlib.reload(mod)
print("✅ spv_tpu package loaded from:", spv_tpu.__file__)

## 3. ユーザ設定パラメータ（`cfg` & TPU `backend`）の編集
`spv/my_geometry.py`・`spv_gpu` と完全互換の構造パラメータと、TPU 専用バックエンド設定を定義します。
- `structure_type`: `6` (Singular Bowtie PhC), `3` (Microdisk), `4` (Rough Microdisk), `5` (Waveguide-coupled Disk), `0` (2D PhC)

In [ ]:
# ==============================================================================
# 3. 計算条件の設定 (cfg & TPU backend) と自動導出パラメータの確認
# ==============================================================================
from spv_tpu.config import get_default_config, derive_parameters, detect_tpu_hardware

hw_info = detect_tpu_hardware()
print("🖥️ Detected Accelerator:", hw_info)

cfg, backend = get_default_config()

# --- 構造と探索条件 (spv_gpu / spv と完全互換) ---
cfg.update(
    structure_type = 6,          # 3:microdisk, 4:rough_disk, 5:wg_disk, 6:singular_bowtie_phc, 0:2D_PhC
    structure      = dict(
        n            = 3.4,      # コア屈折率 (eps_r = 11.56)
        a            = 0.45,     # [um] 格子定数 period
        r_ratio      = 0.26,     # 空気孔半径 r / a
        slab_ratio   = 0.50,     # スラブ厚 slab_t / a
        N_ring       = 8,        # ツイストハニカム格子のリング数
        twist_deg    = 3.5,      # 副格子A/Bの相対ツイスト角 [deg]
        core_r_ratio = 0.55,     # 中心ボウタイ領域の外径半径 core_r / a
        gap_ratio    = 0.02,     # 中心ギャップ幅 g / a
        bowtie_angle = 60.0,     # ボウタイ頂角 [deg]
    ),
    f_range           = (0.55, 0.75),
    parities          = ["+", "-"],
    accuracy          = dict(coarse=2, fine=4),
    quality           = "standard", # "fast" | "standard" | "high"
    n_modes_select    = 3,
    Q_expected        = 1e4,
    run_mode          = "ALL",      # "ALL" | "HARMINV" | "MODE"
    merge_refine_mode = True,       # True で STEP 2.5 と STEP 3 を1回のランに統合
    output_dir        = DEFAULT_OUT_DIR,
)

# --- TPU バックエンド最適化設定 ---
backend.update(
    dtype                 = "float32",  # TPU では float32 推奨
    max_tpu_mem_frac      = 0.75,       # HBM 上限の 75% 以内に res を自動制御
    x64_refine            = False,      # TPU はハードウェア float64 非対応
    allow_tpu_x64         = False,
    chunk_steps           = 2000,       # TPU XLA scan 効率を最大化するチャンク長
    scan_unroll           = 1,
    subpixel_n            = 4,
    anisotropic_averaging = False,
    pml_mode              = "cpml",     # "cpml" (境界スラブ CFS-CPML, 推奨) | "sponge"
    align_tpu_tiles       = True,       # (8 x 128) VPU タイルアラインメント有効化
    sharding              = "auto",     # マルチチップ TPU で自動的に y 軸 shard_map を使用
    xla_cache_dir         = XLA_CACHE_DIR,
    keep_jit_cache        = True,
)

derived = derive_parameters(cfg, backend, hw_info=hw_info)
print("\n=== 自動導出された TPU 計算パラメータ ===")
for k in [
    "fcen", "df", "feat_min", "res_harminv", "res_mode",
    "grid_harminv", "grid_mode", "n_shards_harminv", "n_shards_mode",
    "align_tpu_tiles", "pml_mode", "parities", "t_src_end",
    "T_MAX_harminv", "T_MAX_refine", "T_MAX_mode",
    "est_mem_harminv_gb", "est_mem_harminv_per_dev_gb",
    "est_mem_mode_gb", "est_mem_mode_per_dev_gb", "tpu_per_device_gb", "tpu_mem_gb"
]:
    print(f"  {k:28s}: {derived[k]}")

## 4. 誘電率構造プレビュー（`epchk` 断面確認）
TPU `(Nz, Ny, Nx)` レイアウトで生成された 3D 誘電率分布 $\varepsilon(x,y,z)$ の 3 断面（xy, xz, yz）を確認します。

In [ ]:
# ==============================================================================
# 4. 誘電率構造 (epchk) の事前確認とインライン表示
# ==============================================================================
import gc
from IPython.display import Image, display
from spv_tpu.geometry.structures import build_structure_permittivity
from spv_tpu.pipeline import plot_epchk_slices

pz_preview = derived["parities"][0]
eps_preview, coords_preview, prefix_str = build_structure_permittivity(
    cfg=cfg,
    sp=derived["sp_mode"],
    res=derived["res_mode"],
    pz=pz_preview,
    eps_averaging=True,
    subpixel_n=backend["subpixel_n"],
    grid_shape=derived["grid_mode"],
    layout="zyx",
)

preview_png = "./epchk_tpu_preview.png"
plot_epchk_slices(
    eps_preview,
    coords_preview,
    pz=pz_preview,
    out_png_path=preview_png,
    title=f"Preview: {prefix_str} (res={derived['res_mode']} px/um, grid={derived['grid_mode']}, pz={pz_preview})",
    layout="zyx",
)
display(Image(filename=preview_png))
del eps_preview
gc.collect()

## 5. 全自動パイプライン実行（STEP 1 → STEP 2 → STEP 2.5 → STEP 3）
下のセルを実行すると、TPU 上での FDTD 時間発展・本家 Harminv CLI 解析・モード自動選別・高解像度精密化・2D 断面モードプロファイル生成を一気通貫で実行します。

In [ ]:
# ==============================================================================
# 5. 全自動パイプライン (STEP 1 -> STEP 2 -> STEP 2.5 -> STEP 3) の実行
# ==============================================================================
import os
from IPython.display import Image, display, Markdown
from spv_tpu.pipeline import run_spv_pipeline

pipeline_result = run_spv_pipeline(cfg=cfg, backend=backend)

print(f"\n📂 出力ディレクトリ: {pipeline_result['output_dir']}")
display(Markdown("### 📊 選出・精密化された共振モード一覧 (summary.csv)"))
display(pipeline_result["summary_df"])

for png_path in pipeline_result["harminv_pngs"]:
    display(Markdown(f"#### 📈 Harminv 検出分布: {os.path.basename(png_path)}"))
    display(Image(filename=png_path))

for png_path in pipeline_result["mode_profile_pngs"]:
    display(Markdown(f"#### 🌐 共振モード断面プロファイル: {os.path.basename(png_path)}"))
    display(Image(filename=png_path))

## 6. 段階検証・TPU ベンチマークスイート（V1〜V5 ＋ 鏡像対称 / CPML 検証）
- **V1**: サブピクセル平滑化誘電率の相対 L2 誤差検証（`< 2%`）
- **V4**: 本家 `harminv` CLI の単体精度検証
- **Mirror BC & CFS-CPML**: `pz=+1` (PMC) / `pz=-1` (PEC) 結合と CPML 吸収安定性検証
- **V5**: TPU スループット（Mcells·step/s、初回/ウォーム JIT 時間、推定 HBM 帯域幅 [GB/s]、CPU Meep 比倍率）
- **V2 / V3**: 2D PhC 欠陥共振器および Singular Bowtie PhC（偶・奇両パリティ）の共振抽出テスト

In [ ]:
# ==============================================================================
# 6. 段階検証 (V1, V4, Mirror/CPML, V5 + オプションで V2, V3) の実行
# ==============================================================================
import os
import pandas as pd
from IPython.display import Image, display
from spv_tpu.validate_vs_meep import (
    validate_v1_permittivity,
    validate_v4_harminv_cli,
    validate_mirror_bc_and_cpml,
    validate_v5_speed_benchmark,
    validate_v2_2d_phc,
    validate_v3_type6_bowtie,
)

val_dir = "./spv_tpu_validation"
v1_res = validate_v1_permittivity(out_dir=val_dir)
v4_res = validate_v4_harminv_cli()
bc_res = validate_mirror_bc_and_cpml()
v5_res = validate_v5_speed_benchmark()

def _badge(res):
    return "✅" if res.get("passed") else "❌"

print(f"{_badge(v1_res)} V1 (Permittivity L2 Error < 2%):", v1_res)
if "plot_path" in v1_res and os.path.exists(v1_res["plot_path"]):
    display(Image(filename=v1_res["plot_path"]))

print(f"\n{_badge(v4_res)} V4 (Harminv CLI Unit Test):", v4_res)
print(f"\n{_badge(bc_res)} Mirror BC & CFS-CPML Test:", bc_res)
print(f"\n{_badge(v5_res)} V5 (TPU Throughput & HBM Bandwidth Benchmark):")
display(pd.DataFrame(v5_res["benchmarks"]))

RUN_FULL_V2_V3 = False
if RUN_FULL_V2_V3:
    v2_res = validate_v2_2d_phc(out_dir=val_dir)
    v3_res = validate_v3_type6_bowtie(out_dir=val_dir)
    print(f"\n{_badge(v2_res)} V2 (2D PhC Cavity):", v2_res)
    print(f"\n{_badge(v3_res)} V3 (Type 6 Bowtie Even/Odd):", v3_res)